<a href="https://colab.research.google.com/github/Ezliene/FYP-Solar-Irradience-Prediction/blob/main/solar_isdkh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import numpy as np, pandas as pd
P = '/content/drive/MyDrive/fyp_solar/'

X_train = np.load(P+'train_X_24h_v2.npy')
X_val   = np.load(P+'val_X_24h_v2.npy')
X_test  = np.load(P+'test_X_24h_v2.npy')
y_train = np.load(P+'train_y_24h_v2.npy')
y_val   = np.load(P+'val_y_24h_v2.npy')
y_test  = np.load(P+'test_y_24h_v2.npy')

val_meta  = pd.read_csv(P+'val_meta_24h.xls')
test_meta = pd.read_csv(P+'test_meta_24h.xls')
scaler    = pd.read_csv(P+'solar_scaler_stats.xls')

for n, a in [('X_train',X_train),('X_val',X_val),('X_test',X_test),
             ('y_train',y_train),('y_val',y_val),('y_test',y_test)]:
    print(n, a.shape, a.dtype)

print(val_meta.head()); print(scaler)

Mounted at /content/drive
X_train (2770824, 24, 7) float32
X_val (741034, 24, 7) float32
X_test (735451, 24, 7) float32
y_train (2770824,) float32
y_val (741034,) float32
y_test (735451,) float32
      place             datetime
0  Air Itam  2023-04-13 16:00:00
1  Air Itam  2023-04-13 17:00:00
2  Air Itam  2023-04-13 18:00:00
3  Air Itam  2023-04-13 19:00:00
4  Air Itam  2023-04-13 20:00:00
                 place  solar_mean   solar_std
0             Air Itam  210.325617  286.021714
1          Ampang Jaya  197.509221  265.544312
2           Ayer Keroh  200.795970  270.202260
3   Bandar Puteri Jaya  214.346427  288.503132
4   Bandar Putra Kulai  198.301502  267.783049
..                 ...         ...         ...
62           Tamparuli  223.363465  300.622362
63         Tanjung Aru  207.048226  274.919525
64      Tanjung Bungah  211.956378  285.842560
65               Tawau  205.002165  270.527211
66             Tegupeh  216.625046  291.358189

[67 rows x 3 columns]


In [ ]:
# 1. Is y scaled? (z-scored would be mean≈0, std≈1)
print('y_train mean/std/min/max:', y_train.mean(), y_train.std(), y_train.min(), y_train.max())

# 2. Any NaNs?
print('NaN in X:', np.isnan(X_train[:200000]).any(), '| NaN in y:', np.isnan(y_train).any())

# 3. Meta lines up with y?
print(len(val_meta)==len(y_val), len(test_meta)==len(y_test))

# 4. Every station in meta has scaler stats?
stats = scaler.set_index('place')
print('Missing stations:', set(val_meta.place) - set(stats.index), set(test_meta.place) - set(stats.index))
print('Stations in val/test:', val_meta.place.nunique(), test_meta.place.nunique())

# 5. Helper: scaled -> real W/m²
def to_wm2(y_scaled, meta):
    mu = meta['place'].map(stats['solar_mean']).values
    sd = meta['place'].map(stats['solar_std']).values
    return y_scaled * sd + mu

print(to_wm2(y_val[:5], val_meta.iloc[:5]))   # should look like realistic GHI values

y_train mean/std/min/max: -0.0029063546 0.9985686 -0.7741195 3.1397836
NaN in X: False | NaN in y: False
True True
Missing stations: set() set()
Stations in val/test: 67 67
[ 2.48679993e+02  8.47200068e+01  3.69999926e+00 -1.06054526e-05
 -1.06054526e-05]


In [ ]:
def metrics_table(p, t, meta, day_thr=50):
    df = pd.DataFrame({'place': meta['place'].values, 'p': p, 't': t})
    def m(g):
        e = g.p - g.t
        rmse = np.sqrt((e**2).mean())
        day = g.t > day_thr
        return pd.Series({
            'RMSE': rmse,
            'MAE': e.abs().mean(),
            'R2': 1 - (e**2).sum() / ((g.t - g.t.mean())**2).sum(),
            'nRMSE': rmse / g.t.mean(),
            'MAPE_day%': (e[day].abs() / g.t[day]).mean() * 100,
        })
    per_station = df.groupby('place')[['p','t']].apply(m)
    return per_station, per_station.mean()

# --- Persistence baseline: next hour = last observed GHI ---
cors = [np.corrcoef(X_val[:200000, -1, j], y_val[:200000])[0, 1] for j in range(7)]
GHI_IDX = int(np.argmax(cors))
print(np.round(cors, 3), '-> GHI column:', GHI_IDX)

t_test = to_wm2(y_test, test_meta)
p_pers = np.clip(to_wm2(X_test[:, -1, GHI_IDX], test_meta), 0, None)
ps_pers, overall_pers = metrics_table(p_pers, t_test, test_meta)
print(overall_pers)

[ 0.371  0.176  0.123 -0.348  0.226  0.371  0.937] -> GHI column: 6
RMSE         96.326637
MAE          61.373107
R2            0.876357
nRMSE         0.469623
MAPE_day%    46.658556
dtype: float64


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print(tf.config.list_physical_devices('GPU'))   # should show a GPU
tf.keras.utils.set_random_seed(42)

BATCH = 1024
def make_ds(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(100_000, seed=42)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, shuffle=True)
val_ds   = make_ds(X_val, y_val)

def build_lstm():
    m = models.Sequential([
        layers.Input(shape=(24, 7)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1),
    ], name='lstm')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

lstm = build_lstm()
lstm.summary()

cbs = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss'),
    callbacks.ModelCheckpoint(P+'lstm_24h_best.keras', save_best_only=True, monitor='val_loss'),
]

hist = lstm.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cbs)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,393 (122.63 KB)

 Trainable params: 31,393 (122.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0337 - mae: 0.1068 - val_loss: 0.0181 - val_mae: 0.0776 - learning_rate: 0.0010
Epoch 2/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 42s 16ms/step - loss: 0.0167 - mae: 0.0717 - val_loss: 0.0207 - val_mae: 0.0857 - learning_rate: 0.0010
Epoch 3/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 81s 15ms/step - loss: 0.0153 - mae: 0.0661 - val_loss: 0.0210 - val_mae: 0.0961 - learning_rate: 0.0010
Epoch 4/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 43s 16ms/step - loss: 0.0146 - mae: 0.0630 - val_loss: 0.0244 - val_mae: 0.1000 - learning_rate: 5.0000e-04
Epoch 5/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 45s 17ms/step - loss: 0.0144 - mae: 0.0621 - val_loss: 0.0254 - val_mae: 0.1016 - learning_rate: 5.0000e-04
Epoch 6/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 44s 16ms/step - loss: 0.0141 - mae: 0.0610 - val_loss: 0.0287 - val_mae: 0.1041 - learning_rate: 2.5000e-04


In [ ]:
def evaluate(model, X, y, meta):
    pred = model.predict(X, batch_size=4096).ravel()
    p = np.clip(to_wm2(pred, meta), 0, None)
    t = to_wm2(y, meta)
    rmse = np.sqrt(np.mean((p-t)**2))
    mae  = np.mean(np.abs(p-t))
    r2   = 1 - np.sum((p-t)**2)/np.sum((t-t.mean())**2)
    nrmse = rmse / t.mean()
    return dict(RMSE=rmse, MAE=mae, R2=r2, nRMSE=nrmse), p, t

res_lstm, p_lstm, t_test = evaluate(lstm, X_test, y_test, test_meta)
print(res_lstm)

180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step
{'RMSE': np.float64(36.91919101214001), 'MAE': np.float64(21.492255289182204), 'R2': np.float64(0.9823911136041196), 'nRMSE': np.float64(0.17884723985066103)}


In [ ]:
j = GHI_IDX
# Leak if ~0: the window's last step already contains the target
print('own y vs last step:', np.median(np.abs(X_val[:, -1, j] - y_val)))
# Should be ~0: the next sample's last step equals this sample's target
print('next sample vs y  :', np.median(np.abs(X_val[1:, -1, j] - y_val[:-1])))

own y vs last step: 0.06529653
next sample vs y  : 0.0


In [ ]:
ps_lstm, overall_lstm = metrics_table(p_lstm, t_test, test_meta)
print(pd.concat([overall_lstm.rename('LSTM'), overall_pers.rename('Persistence')], axis=1))

                LSTM  Persistence
RMSE       36.145805    96.326637
MAE        21.153903    61.373107
R2          0.982454     0.876357
nRMSE       0.176571     0.469623
MAPE_day%  12.090046    46.658556


In [ ]:
# 1. Shuffle once across all samples
perm = np.random.RandomState(42).permutation(len(X_train))
X_train, y_train = X_train[perm], y_train[perm]

# 2. Rebuild datasets (no more small shuffle buffer)
train_ds = make_ds(X_train, y_train, shuffle=False)
val_ds   = make_ds(X_val, y_val)

# 3. Model matching Table 3-2
def build_lstm():
    m = models.Sequential([
        layers.Input(shape=(24, 7)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name='lstm_v2')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

tf.keras.utils.set_random_seed(42)
lstm = build_lstm()
cbs = [
    callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss'),
    callbacks.ModelCheckpoint(P+'lstm_24h_v2_best.keras', save_best_only=True, monitor='val_loss'),
]
hist = lstm.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cbs)

Epoch 1/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 14ms/step - loss: 0.0387 - mae: 0.1232 - val_loss: 0.0139 - val_mae: 0.0635 - learning_rate: 0.0010
Epoch 2/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 36s 13ms/step - loss: 0.0233 - mae: 0.1001 - val_loss: 0.0132 - val_mae: 0.0595 - learning_rate: 0.0010
Epoch 3/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 14ms/step - loss: 0.0226 - mae: 0.0979 - val_loss: 0.0133 - val_mae: 0.0592 - learning_rate: 0.0010
Epoch 4/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 37s 13ms/step - loss: 0.0222 - mae: 0.0968 - val_loss: 0.0133 - val_mae: 0.0608 - learning_rate: 0.0010
Epoch 5/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 37s 14ms/step - loss: 0.0218 - mae: 0.0956 - val_loss: 0.0128 - val_mae: 0.0581 - learning_rate: 5.0000e-04
Epoch 6/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 14ms/step - loss: 0.0216 - mae: 0.0953 - val_loss: 0.0127 - val_mae: 0.0571 - learning_rate: 5.0000e-04
Epoch 7/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 37s 14ms/step - loss: 0.0216 - mae: 0.0950 - val_loss: 0.0125 - val_mae: 0

In [ ]:
def evaluate(model, X, y, meta):
    pred = model.predict(X, batch_size=4096).ravel()
    p = np.clip(to_wm2(pred, meta), 0, None)
    t = to_wm2(y, meta)
    rmse = np.sqrt(np.mean((p-t)**2))
    mae  = np.mean(np.abs(p-t))
    r2   = 1 - np.sum((p-t)**2)/np.sum((t-t.mean())**2)
    nrmse = rmse / t.mean()
    return dict(RMSE=rmse, MAE=mae, R2=r2, nRMSE=nrmse), p, t

res_lstm, p_lstm, t_test = evaluate(lstm, X_test, y_test, test_meta)
print(res_lstm)

180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step
{'RMSE': np.float64(30.34956470565395), 'MAE': np.float64(14.754745002761357), 'R2': np.float64(0.9881003969983382), 'nRMSE': np.float64(0.14702206980890775)}


In [ ]:
ps_lstm, overall_lstm = metrics_table(p_lstm, t_test, test_meta)
print(pd.concat([overall_lstm.rename('LSTM'), overall_pers.rename('Persistence')], axis=1))

                LSTM  Persistence
RMSE       29.616852    96.326637
MAE        14.639258    61.373107
R2          0.988156     0.876357
nRMSE       0.144823     0.469623
MAPE_day%   8.733712    46.658556


In [ ]:
res = pd.concat([overall_lstm.rename('LSTM'), overall_pers.rename('Persistence')], axis=1)
res.to_csv(P+'results_overall_so_far.csv')
ps_lstm.to_csv(P+'per_station_lstm_v2.csv')
ps_pers.to_csv(P+'per_station_persistence.csv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers, models, callbacks
P = '/content/drive/MyDrive/fyp_solar/'
print(sorted(os.listdir(P)))          # <- what's saved in your folder

# --- data ---
X_train = np.load(P+'train_X_24h_v2.npy'); y_train = np.load(P+'train_y_24h_v2.npy')
X_val   = np.load(P+'val_X_24h_v2.npy');   y_val   = np.load(P+'val_y_24h_v2.npy')
X_test  = np.load(P+'test_X_24h_v2.npy');  y_test  = np.load(P+'test_y_24h_v2.npy')
val_meta  = pd.read_csv(P+'val_meta_24h.xls')
test_meta = pd.read_csv(P+'test_meta_24h.xls')
stats = pd.read_csv(P+'solar_scaler_stats.xls').set_index('place')

# --- helpers ---
def to_wm2(y_scaled, meta):
    mu = meta['place'].map(stats['solar_mean']).values
    sd = meta['place'].map(stats['solar_std']).values
    return y_scaled * sd + mu

def metrics_table(p, t, meta, day_thr=50):
    df = pd.DataFrame({'place': meta['place'].values, 'p': p, 't': t})
    def m(g):
        e = g.p - g.t
        rmse = np.sqrt((e**2).mean())
        day = g.t > day_thr
        return pd.Series({
            'RMSE': rmse,
            'MAE': e.abs().mean(),
            'R2': 1 - (e**2).sum() / ((g.t - g.t.mean())**2).sum(),
            'nRMSE': rmse / g.t.mean(),
            'MAPE_day%': (e[day].abs() / g.t[day]).mean() * 100,
        })
    per_station = df.groupby('place')[['p','t']].apply(m)
    return per_station, per_station.mean()

# --- shuffle once + datasets ---
perm = np.random.RandomState(42).permutation(len(X_train))
X_train, y_train = X_train[perm], y_train[perm]
BATCH = 1024
def make_ds(X, y):
    return tf.data.Dataset.from_tensor_slices((X, y)).batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds = make_ds(X_train, y_train), make_ds(X_val, y_val)

# --- reload LSTM + persistence, rebuild the comparison ---
GHI_IDX = 6
t_test = to_wm2(y_test, test_meta)

lstm = models.load_model(P+'lstm_24h_v2_best.keras')
p_lstm = np.clip(to_wm2(lstm.predict(X_test, batch_size=4096).ravel(), test_meta), 0, None)
ps_lstm, overall_lstm = metrics_table(p_lstm, t_test, test_meta)

p_pers = np.clip(to_wm2(X_test[:, -1, GHI_IDX], test_meta), 0, None)
ps_pers, overall_pers = metrics_table(p_pers, t_test, test_meta)

print(pd.concat([overall_lstm.rename('LSTM'), overall_pers.rename('Persistence')], axis=1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['lstm_24h_best.keras', 'lstm_24h_v2_best.keras', 'per_station_lstm_v2.csv', 'per_station_persistence.csv', 'results_overall_so_far.csv', 'solar_scaler_stats.xls', 'test_X_24h_v2.npy', 'test_meta_24h.xls', 'test_y_24h_v2.npy', 'train_X_24h_v2.npy', 'train_y_24h_v2.npy', 'val_X_24h_v2.npy', 'val_meta_24h.xls', 'val_y_24h_v2.npy']
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step
                LSTM  Persistence
RMSE       29.616852    96.326637
MAE        14.639258    61.373107
R2          0.988156     0.876357
nRMSE       0.144823     0.469623
MAPE_day%   8.733712    46.658556


In [ ]:
def build_cnn_bilstm():
    m = models.Sequential([
        layers.Input(shape=(24, 7)),
        layers.Conv1D(64, kernel_size=3, padding='same', activation='relu'),
        layers.MaxPooling1D(pool_size=2),                # 24 -> 12 steps
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name='cnn_bilstm')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

tf.keras.utils.set_random_seed(42)
cnn_bilstm = build_cnn_bilstm()
cnn_bilstm.summary()

cbs = [
    callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-5, monitor='val_loss'),
    callbacks.ModelCheckpoint(P+'cnn_bilstm_24h_best.keras', save_best_only=True, monitor='val_loss'),
]
hist2 = cnn_bilstm.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cbs)

Model: "cnn_bilstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 24, 64)         │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,585 (264.00 KB)

 Trainable params: 67,585 (264.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - loss: 0.0262 - mae: 0.0985 - val_loss: 0.0139 - val_mae: 0.0662 - learning_rate: 0.0010
Epoch 2/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 34s 13ms/step - loss: 0.0169 - mae: 0.0782 - val_loss: 0.0133 - val_mae: 0.0616 - learning_rate: 0.0010
Epoch 3/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 33s 12ms/step - loss: 0.0162 - mae: 0.0755 - val_loss: 0.0130 - val_mae: 0.0587 - learning_rate: 0.0010
Epoch 4/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 33s 12ms/step - loss: 0.0158 - mae: 0.0741 - val_loss: 0.0130 - val_mae: 0.0606 - learning_rate: 0.0010
Epoch 5/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - loss: 0.0156 - mae: 0.0732 - val_loss: 0.0126 - val_mae: 0.0558 - learning_rate: 0.0010
Epoch 6/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 34s 13ms/step - loss: 0.0155 - mae: 0.0726 - val_loss: 0.0129 - val_mae: 0.0564 - learning_rate: 0.0010
Epoch 7/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 33s 12ms/step - loss: 0.0154 - mae: 0.0722 - val_loss: 0.0127 - val_mae: 0.0564 - 

In [ ]:
pred = cnn_bilstm.predict(X_test, batch_size=4096).ravel()
p_cnn = np.clip(to_wm2(pred, test_meta), 0, None)
ps_cnn, overall_cnn = metrics_table(p_cnn, t_test, test_meta)

res = pd.concat([overall_lstm.rename('LSTM'),
                 overall_cnn.rename('CNN-BiLSTM'),
                 overall_pers.rename('Persistence')], axis=1)
print(res)

# save so a disconnect can't lose it
res.to_csv(P+'results_overall_v2.csv')
ps_cnn.to_csv(P+'per_station_cnn_bilstm.csv')

180/180 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step
                LSTM  CNN-BiLSTM  Persistence
RMSE       29.616852   28.945891    96.326637
MAE        14.639258   14.104123    61.373107
R2          0.988156    0.988691     0.876357
nRMSE       0.144823    0.141529     0.469623
MAPE_day%   8.733712    8.381789    46.658556


In [ ]:
cmp = pd.DataFrame({
    'LSTM': ps_lstm['RMSE'],
    'CNN-BiLSTM': ps_cnn['RMSE'],
    'Persistence': ps_pers['RMSE'],
})
cmp['CNN wins'] = cmp['CNN-BiLSTM'] < cmp['LSTM']

print('CNN-BiLSTM beats LSTM at', cmp['CNN wins'].sum(), 'of', len(cmp), 'stations')
print('\nBest 5 stations:');  print(cmp.sort_values('CNN-BiLSTM').head(5).round(1))
print('\nWorst 5 stations:'); print(cmp.sort_values('CNN-BiLSTM').tail(5).round(1))
print('\nnRMSE spread across stations:'); print(ps_cnn['nRMSE'].describe().round(3))

cmp.to_csv(P+'per_station_rmse_compare.csv')

CNN-BiLSTM beats LSTM at 66 of 67 stations

Best 5 stations:
              LSTM  CNN-BiLSTM  Persistence  CNN wins
place                                                
Muallim       24.3        23.6         97.0      True
Sejingkat     27.0        25.4         92.2      True
Nilai Impian  25.8        25.4        100.8      True
Gita Bayu     26.4        25.5         94.9      True
Sungai Ruan   26.6        25.9         91.6      True

Worst 5 stations:
                    LSTM  CNN-BiLSTM  Persistence  CNN wins
place                                                      
Tanjung Bungah      32.7        32.0         97.1      True
Johor Bahru         32.4        32.3         94.3      True
Sandakan            33.3        33.1         99.8      True
Taman Kulai Besar   34.3        33.5         94.8      True
Bandar Putra Kulai  34.4        33.9         93.7      True

nRMSE spread across stations:
count    67.000
mean      0.142
std       0.014
min       0.110
25%       0.132
50%       0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import models
P = '/content/drive/MyDrive/fyp_solar/'

# --- test data only ---
X_test = np.load(P+'test_X_24h_v2.npy')
y_test = np.load(P+'test_y_24h_v2.npy')

def read_any(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_excel(path)

test_meta = read_any(P+'test_meta_24h.xls')
stats     = read_any(P+'solar_scaler_stats.xls').set_index('place')

# --- helpers ---
def to_wm2(y_scaled, meta):
    mu = meta['place'].map(stats['solar_mean']).values
    sd = meta['place'].map(stats['solar_std']).values
    return y_scaled * sd + mu

def metrics_table(p, t, meta, day_thr=50):
    df = pd.DataFrame({'place': meta['place'].values, 'p': p, 't': t})
    def m(g):
        e = g.p - g.t
        rmse = np.sqrt((e**2).mean())
        day = g.t > day_thr
        return pd.Series({
            'RMSE': rmse,
            'MAE': e.abs().mean(),
            'R2': 1 - (e**2).sum() / ((g.t - g.t.mean())**2).sum(),
            'nRMSE': rmse / g.t.mean(),
            'MAPE_day%': (e[day].abs() / g.t[day]).mean() * 100,
        })
    per_station = df.groupby('place')[['p','t']].apply(m)
    return per_station, per_station.mean()

# --- models + predictions ---
GHI_IDX = 6
t_test = to_wm2(y_test, test_meta)

lstm       = models.load_model(P+'lstm_24h_v2_best.keras')
cnn_bilstm = models.load_model(P+'cnn_bilstm_24h_best.keras')

p_lstm = np.clip(to_wm2(lstm.predict(X_test, batch_size=4096).ravel(), test_meta), 0, None)
p_cnn  = np.clip(to_wm2(cnn_bilstm.predict(X_test, batch_size=4096).ravel(), test_meta), 0, None)
p_pers = np.clip(to_wm2(X_test[:, -1, GHI_IDX], test_meta), 0, None)

print('Ready ✅', X_test.shape, len(test_meta))

Mounted at /content/drive
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step
180/180 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step
Ready ✅ (735451, 24, 7) 735451


In [ ]:
nn = test_meta[['place','datetime']].copy()
nn['datetime'] = pd.to_datetime(nn['datetime'])
nn['y_true'] = t_test
nn['LSTM'], nn['CNN-BiLSTM'], nn['Persistence'] = p_lstm, p_cnn, p_pers

xg = pd.read_csv(P+'test_predictions_real_units_v2.xls')
xg['datetime'] = pd.to_datetime(xg['datetime'])
xg = xg.rename(columns={'y_pred_real':'XGBoost', 'y_true_real':'y_true_xgb'})

m = nn.merge(xg[['place','datetime','XGBoost','y_true_xgb']], on=['place','datetime'], how='inner')
print('NN rows:', len(nn), '| XGB rows:', len(xg), '| shared:', len(m))
print('max |true diff| (should be ~0):', (m['y_true'] - m['y_true_xgb']).abs().max())

final = pd.DataFrame({
    name: metrics_table(m[name].values, m['y_true'].values, m[['place']])[1]
    for name in ['CNN-BiLSTM','XGBoost','LSTM','Persistence']
}).round(4)
print(final)
final.to_csv(P+'final_comparison_67stations.csv')

NN rows: 735451 | XGB rows: 749012 | shared: 735451
max |true diff| (should be ~0): 3.5143326385878026e-05
           CNN-BiLSTM  XGBoost     LSTM  Persistence
RMSE          28.9459  29.4623  29.6169      96.3266
MAE           14.1041  14.1383  14.6393      61.3731
R2             0.9887   0.9883   0.9882       0.8764
nRMSE          0.1415   0.1441   0.1448       0.4696
MAPE_day%      8.3818   8.5078   8.7337      46.6586


In [ ]:
yrs = pd.read_csv(P+'station_years.xls').set_index('place')['years']
long_st = set(yrs[yrs >= 15].index) - {'Mersing'}
n_long, n_short = len(long_st), len(yrs) - len(long_st)     # should be 23 and 44

rows = {}
for name in ['CNN-BiLSTM','XGBoost','LSTM','Persistence']:
    ps, _ = metrics_table(m[name].values, m['y_true'].values, m[['place']])
    ps['group'] = np.where(ps.index.isin(long_st),
                           f'long ({n_long} stations)', f'short ({n_short} stations)')
    g = ps.groupby('group')[['RMSE','MAE','R2','nRMSE','MAPE_day%']].mean()
    for grp in g.index:
        rows[(grp, name)] = g.loc[grp]

res = pd.DataFrame(rows).T.round(3)
print(res)
res.to_csv(P+'comparison_by_record_length.csv')

for name in ['CNN-BiLSTM','XGBoost','LSTM','Persistence']:
    e = m[name] - m['y_true']
    print(f"{name:12s} pooled RMSE {np.sqrt((e**2).mean()):.2f} | MAE {e.abs().mean():.2f}")

                                   RMSE     MAE     R2  nRMSE  MAPE_day%
long (23 stations)  CNN-BiLSTM   30.145  14.422  0.988  0.146      8.685
short (44 stations) CNN-BiLSTM   28.319  13.938  0.989  0.139      8.224
long (23 stations)  XGBoost      30.624  14.528  0.988  0.149      9.044
short (44 stations) XGBoost      28.855  13.935  0.989  0.142      8.227
long (23 stations)  LSTM         30.558  14.821  0.988  0.148      8.971
short (44 stations) LSTM         29.125  14.544  0.988  0.143      8.610
long (23 stations)  Persistence  97.875  62.066  0.877  0.474     46.708
short (44 stations) Persistence  95.517  61.011  0.876  0.467     46.633
CNN-BiLSTM   pooled RMSE 29.86 | MAE 14.34
XGBoost      pooled RMSE 30.35 | MAE 14.41
LSTM         pooled RMSE 30.35 | MAE 14.75
Persistence  pooled RMSE 97.60 | MAE 61.94


In [ ]:
ps_cnn, _ = metrics_table(m['CNN-BiLSTM'].values, m['y_true'].values, m[['place']])
lg = ps_cnn[ps_cnn.index.isin(long_st)].sort_values('RMSE')
pick = [lg.index[0], lg.index[len(lg)//2], lg.index[-1]]
print(pick)
print(lg[['RMSE','MAE','R2']].round(2).loc[pick])

['Kuantan', 'Sabu', 'Bandar Putra Kulai']
                     RMSE    MAE    R2
place                                 
Kuantan             27.86  13.25  0.99
Sabu                29.95  14.53  0.99
Bandar Putra Kulai  33.89  15.67  0.98


In [ ]:
bs = pd.read_csv(P+'baseline_stations_raw.xls', parse_dates=['datetime'])
print(bs.groupby(['place','split'])['datetime'].agg(['min','max','count']))

# test rows the neural-network comparison table has for these stations
print(m[m['place'].isin(bs['place'].unique())].groupby('place').size())

                                         min                 max   count
place              split                                                
Bandar Putra Kulai test  2021-07-03 16:00:00 2024-08-31 15:00:00   27559
                   train 2001-01-01 00:00:00 2018-02-03 21:00:00  128605
                   val   2018-02-03 22:00:00 2021-07-03 15:00:00   27558
Kuantan            test  2021-10-10 19:00:00 2024-08-31 15:00:00   25216
                   train 2001-01-11 13:00:00 2018-09-20 03:00:00  117672
                   val   2018-09-20 04:00:00 2021-10-10 18:00:00   25215
Sabu               test  2021-09-09 16:00:00 2024-08-31 15:00:00   25635
                   train 2001-03-09 22:00:00 2018-07-26 18:00:00  119628
                   val   2018-07-26 19:00:00 2021-09-09 15:00:00   25635
place
Bandar Putra Kulai    27063
Kuantan               24887
Sabu                  24467
dtype: int64


In [ ]:
import warnings, time
warnings.filterwarnings('ignore')

bs = pd.read_csv(P+'baseline_stations_raw.xls', parse_dates=['datetime'])
STATIONS = ['Kuantan', 'Sabu', 'Bandar Putra Kulai']

def get_series(place):
    d = bs[bs['place'] == place].sort_values('datetime')
    d = d[~d['datetime'].duplicated()]
    s = d.set_index('datetime')['solar_radiation'].asfreq('h')   # hourly, NaN where missing
    return d, s

for st in STATIONS:
    d, s = get_series(st)
    print(st, len(s), 'hours,', int(s.isna().sum()), 'missing')

Kuantan 207195 hours, 39092 missing
Sabu 205818 hours, 34920 missing
Bandar Putra Kulai 207448 hours, 23726 missing


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_out = []
for st in STATIONS:
    t0 = time.time()
    d, s = get_series(st)
    split = d.set_index('datetime')['split']
    tr_end   = split[split == 'train'].index.max()
    te_start = split[split == 'test'].index.min()
    te       = split[split == 'test'].index

    fit_part = s[tr_end - pd.Timedelta(days=120): tr_end] / 100       # scaled for stability
    sm = SARIMAX(fit_part, order=(1,0,1), seasonal_order=(1,0,1,24),
                 enforce_stationarity=False, enforce_invertibility=False)
    sfit = sm.fit(disp=False, maxiter=50)

    ext = s[te_start - pd.Timedelta(days=14):] / 100                  # 2 weeks of warm-up + test
    sext = sfit.apply(ext, refit=False)
    pred = sext.get_prediction().predicted_mean * 100                 # one-step-ahead

    sarima_out.append(pd.DataFrame({'place': st, 'datetime': te,
                                    'SARIMA': np.clip(pred.reindex(te).values, 0, None)}))
    print(st, 'done in', round(time.time() - t0), 's')

sarima = pd.concat(sarima_out)
sarima.to_csv(P+'sarima_preds.csv', index=False)

Kuantan done in 8 s
Sabu done in 8 s


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


Bandar Putra Kulai done in 10 s


In [ ]:
!pip -q install prophet
from prophet import Prophet
import logging
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
logging.getLogger('prophet').setLevel(logging.ERROR)

prophet_out = []
for st in STATIONS:
    t0 = time.time()
    d = bs[bs['place'] == st].sort_values('datetime')
    train_df = (d[d['split'].isin(['train','val'])][['datetime','solar_radiation']]
                .rename(columns={'datetime':'ds', 'solar_radiation':'y'}))
    test_df = d[d['split'] == 'test'][['datetime']].rename(columns={'datetime':'ds'})

    pm = Prophet(daily_seasonality=False, weekly_seasonality=False, yearly_seasonality=True)
    pm.add_seasonality(name='daily', period=1, fourier_order=10)     # sharper daily curve
    pm.fit(train_df)
    fc = pm.predict(test_df)

    prophet_out.append(pd.DataFrame({'place': st, 'datetime': test_df['ds'].values,
                                     'Prophet': np.clip(fc['yhat'].values, 0, None)}))
    print(st, 'done in', round(time.time() - t0), 's')

prophet = pd.concat(prophet_out)
prophet.to_csv(P+'prophet_preds.csv', index=False)

Kuantan done in 50 s
Sabu done in 96 s
Bandar Putra Kulai done in 69 s


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
P = '/content/drive/MyDrive/fyp_solar/'
print([f for f in sorted(os.listdir(P)) if any(k in f for k in ['sarima','prophet','baseline','test_preds'])])

Mounted at /content/drive
['baseline_stations_raw.xls', 'prophet_preds.csv', 'sarima_preds.csv']


In [ ]:
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import models
P = '/content/drive/MyDrive/fyp_solar/'
STATIONS = ['Kuantan', 'Sabu', 'Bandar Putra Kulai']

# ---------- helpers ----------
def read_any(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_excel(path)

stats = read_any(P+'solar_scaler_stats.xls').set_index('place')

def to_wm2(y_scaled, meta):
    mu = meta['place'].map(stats['solar_mean']).values
    sd = meta['place'].map(stats['solar_std']).values
    return y_scaled * sd + mu

def metrics_table(p, t, meta, day_thr=50):
    df = pd.DataFrame({'place': meta['place'].values, 'p': p, 't': t})
    def f(g):
        e = g.p - g.t
        rmse = np.sqrt((e**2).mean())
        day = g.t > day_thr
        return pd.Series({
            'RMSE': rmse,
            'MAE': e.abs().mean(),
            'R2': 1 - (e**2).sum() / ((g.t - g.t.mean())**2).sum(),
            'nRMSE': rmse / g.t.mean(),
            'MAPE_day%': (e[day].abs() / g.t[day]).mean() * 100,
        })
    ps = df.groupby('place')[['p','t']].apply(f)
    return ps, ps.mean()

# ---------- neural-net + XGBoost predictions ----------
X_test = np.load(P+'test_X_24h_v2.npy')
y_test = np.load(P+'test_y_24h_v2.npy')
test_meta = read_any(P+'test_meta_24h.xls')
t_test = to_wm2(y_test, test_meta)

lstm       = models.load_model(P+'lstm_24h_v2_best.keras')
cnn_bilstm = models.load_model(P+'cnn_bilstm_24h_best.keras')
p_lstm = np.clip(to_wm2(lstm.predict(X_test, batch_size=4096).ravel(), test_meta), 0, None)
p_cnn  = np.clip(to_wm2(cnn_bilstm.predict(X_test, batch_size=4096).ravel(), test_meta), 0, None)
p_pers = np.clip(to_wm2(X_test[:, -1, 6], test_meta), 0, None)

nn = test_meta[['place','datetime']].copy()
nn['datetime'] = pd.to_datetime(nn['datetime'])
nn['y_true'] = t_test
nn['LSTM'], nn['CNN-BiLSTM'], nn['Persistence'] = p_lstm, p_cnn, p_pers

xg = pd.read_csv(P+'test_predictions_real_units_v2.xls')
xg['datetime'] = pd.to_datetime(xg['datetime'])
xg = xg.rename(columns={'y_pred_real':'XGBoost', 'y_true_real':'y_true_xgb'})
m = nn.merge(xg[['place','datetime','XGBoost','y_true_xgb']], on=['place','datetime'], how='inner')
m.to_pickle(P+'test_preds_all_models.pkl')        # saved for next time
print('m rows:', len(m))

# ---------- add SARIMA + Prophet and compare ----------
sarima  = pd.read_csv(P+'sarima_preds.csv');  sarima['datetime']  = pd.to_datetime(sarima['datetime'])
prophet = pd.read_csv(P+'prophet_preds.csv'); prophet['datetime'] = pd.to_datetime(prophet['datetime'])

m3 = (m[m['place'].isin(STATIONS)]
        .merge(sarima,  on=['place','datetime'], how='inner')
        .merge(prophet, on=['place','datetime'], how='inner')
        .dropna(subset=['SARIMA','Prophet']))
print(len(m3), 'shared rows')

models_ = ['CNN-BiLSTM','XGBoost','LSTM','Persistence','SARIMA','Prophet']
final3 = pd.DataFrame({n: metrics_table(m3[n].values, m3['y_true'].values, m3[['place']])[1]
                       for n in models_}).round(3)
print(final3)

per = pd.DataFrame({n: metrics_table(m3[n].values, m3['y_true'].values, m3[['place']])[0]['RMSE']
                    for n in models_}).round(1)
print(per)
final3.to_csv(P+'baseline_comparison_3stations.csv')

180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step
m rows: 735451
76417 shared rows
           CNN-BiLSTM  XGBoost    LSTM  Persistence  SARIMA  Prophet
RMSE           30.564   31.699  30.993       94.403  35.771   86.054
MAE            14.483   14.992  15.007       59.487  18.361   45.617
R2              0.987    0.986   0.986        0.876   0.982    0.896
nRMSE           0.156    0.162   0.158        0.481   0.183    0.439
MAPE_day%       9.444   10.380   9.712       47.018  14.353   33.051
                    CNN-BiLSTM  XGBoost  LSTM  Persistence  SARIMA  Prophet
place                                                                      
Bandar Putra Kulai        33.9     34.9  34.4         93.7    39.8     93.6
Kuantan                   27.9     28.3  28.3         95.2    32.9     90.6
Sabu                      29.9     31.8  30.3         94.4    34.5     73.9


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import glob, os, shutil

W = 48                                   # change to 72 for the second run
DRIVE = '/content/drive/MyDrive/fyp_solar/'   # your folder
LOCAL = '/content/data'
os.makedirs(LOCAL, exist_ok=True)

def get(name):
    hits = glob.glob(f'{DRIVE}/{name}*')
    assert hits, f"not found: {name}"
    dst = f'{LOCAL}/{os.path.basename(hits[0])}'
    if not os.path.exists(dst):
        shutil.copy(hits[0], dst)
    return dst

paths = {k: get(n) for k, n in {
    'Xtr': f'train_X_{W}h_v2.npy', 'ytr': f'train_y_{W}h_v2.npy',
    'Xva': f'val_X_{W}h_v2.npy',   'yva': f'val_y_{W}h_v2.npy',
    'Xte': f'test_X_{W}h_v2.npy',  'yte': f'test_y_{W}h_v2.npy',
    'mte': f'test_meta_{W}h',      'stats': 'solar_scaler_stats'}.items()}
print("copied")

copied


In [ ]:
import numpy as np, pandas as pd
from tensorflow import keras
keras.utils.set_random_seed(42)

# mmap = read from disk batch by batch, so the big 72h arrays don't fill Colab's ~12 GB RAM
Xtr, ytr = np.load(paths['Xtr'], mmap_mode='r'), np.load(paths['ytr'])
Xva, yva = np.load(paths['Xva'], mmap_mode='r'), np.load(paths['yva'])
Xte, yte = np.load(paths['Xte'], mmap_mode='r'), np.load(paths['yte'])
print(Xtr.shape, Xva.shape, Xte.shape)

class Windows(keras.utils.Sequence):
    def __init__(self, X, y, bs=256, shuffle=False):
        super().__init__()
        self.X, self.y, self.bs, self.shuffle = X, y, bs, shuffle
        self.idx = np.arange(len(X))
        self.on_epoch_end()
    def __len__(self):
        return int(np.ceil(len(self.X) / self.bs))
    def __getitem__(self, i):
        b = np.sort(self.idx[i * self.bs:(i + 1) * self.bs])
        return np.asarray(self.X[b]), np.asarray(self.y[b])
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.idx)

(2395891, 48, 7) (721281, 48, 7) (715219, 48, 7)


In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(W, 7)),
    keras.layers.LSTM(64),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')

early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(Windows(Xtr, ytr, 256, shuffle=True), validation_data=Windows(Xva, yva, 1024),
          epochs=20, callbacks=[early], verbose=2)
model.save(f'{DRIVE}/lstm_{W}h_baseline.keras')

Epoch 1/20
9359/9359 - 91s - 10ms/step - loss: 0.0233 - val_loss: 0.0133
Epoch 2/20
9359/9359 - 68s - 7ms/step - loss: 0.0185 - val_loss: 0.0129
Epoch 3/20
9359/9359 - 67s - 7ms/step - loss: 0.0181 - val_loss: 0.0135
Epoch 4/20
9359/9359 - 69s - 7ms/step - loss: 0.0178 - val_loss: 0.0129
Epoch 5/20
9359/9359 - 69s - 7ms/step - loss: 0.0176 - val_loss: 0.0127
Epoch 6/20
9359/9359 - 68s - 7ms/step - loss: 0.0175 - val_loss: 0.0124
Epoch 7/20
9359/9359 - 67s - 7ms/step - loss: 0.0174 - val_loss: 0.0125
Epoch 8/20
9359/9359 - 67s - 7ms/step - loss: 0.0173 - val_loss: 0.0124
Epoch 9/20
9359/9359 - 69s - 7ms/step - loss: 0.0172 - val_loss: 0.0126
Epoch 10/20
9359/9359 - 68s - 7ms/step - loss: 0.0172 - val_loss: 0.0123
Epoch 11/20
9359/9359 - 66s - 7ms/step - loss: 0.0171 - val_loss: 0.0123
Epoch 12/20
9359/9359 - 67s - 7ms/step - loss: 0.0171 - val_loss: 0.0121
Epoch 13/20
9359/9359 - 68s - 7ms/step - loss: 0.0170 - val_loss: 0.0123
Epoch 14/20
9359/9359 - 68s - 7ms/step - loss: 0.0170 - val

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pred = model.predict(Windows(Xte, yte, 4096), verbose=0).ravel()
df = pd.read_csv(paths['mte'])
df['pred_s'], df['true_s'] = pred, yte
df = df.merge(pd.read_csv(paths['stats']), on='place', how='left')
df['pred'] = df['pred_s'] * df['solar_std'] + df['solar_mean']
df['true'] = df['true_s'] * df['solar_std'] + df['solar_mean']

rmse = np.sqrt(mean_squared_error(df['true'], df['pred']))
mae = mean_absolute_error(df['true'], df['pred'])
day = df['true'] > 10
mape = (np.abs(df.loc[day, 'true'] - df.loc[day, 'pred']) / df.loc[day, 'true']).mean() * 100
print(f"LSTM {W}h | RMSE {rmse:.2f} | MAE {mae:.2f} | R2 {r2_score(df['true'], df['pred']):.4f} "
      f"| nRMSE {rmse / df['true'].mean() * 100:.2f}% | MAPE {mape:.2f}%")

df.to_csv(f'{DRIVE}/lstm_{W}h_test_predictions.csv', index=False)

LSTM 48h | RMSE 30.30 | MAE 15.14 | R2 0.9881 | nRMSE 14.68% | MAPE 11.52%


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os

folder = '/content/drive/MyDrive/fyp_solar'

print("Does the FYP folder exist?", os.path.exists(folder))

if os.path.exists(folder):
    print("\nFiles inside fyp_solar:\n")
    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)

        if os.path.isfile(path):
            size_mb = os.path.getsize(path) / (1024 ** 2)
            print(f"{name}  |  {size_mb:.2f} MB")

        elif os.path.isdir(path):
            print(f"[FOLDER] {name}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Does the FYP folder exist? True

Files inside fyp_solar:

baseline_comparison_3stations.csv  |  0.00 MB
baseline_stations_raw.xls  |  21.31 MB
cnn_bilstm_24h_best.keras  |  0.82 MB
comparison_by_record_length.csv  |  0.00 MB
final_comparison_67stations.csv  |  0.00 MB
lstm_24h_best.keras  |  0.40 MB
lstm_24h_v2_best.keras  |  0.39 MB
lstm_48h_baseline.keras  |  0.24 MB
lstm_48h_test_predictions.csv  |  88.67 MB
per_station_cnn_bilstm.csv  |  0.01 MB
per_station_lstm_v2.csv  |  0.01 MB
per_station_persistence.csv  |  0.01 MB
per_station_rmse_compare.csv  |  0.00 MB
prophet_preds.csv  |  3.41 MB
results_overall_so_far.csv  |  0.00 MB
results_overall_v2.csv  |  0.00 MB
sarima_preds.csv  |  3.45 MB
solar_scaler_stats.xls  |  0.00 MB
station_years.xls  |  0.00 MB
test_X_24h_v2.npy  |  471.33 MB
test_X_48h_v2.npy  |  916.72 MB
test_meta_24h.xls  |  23.25 MB
test_me

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import glob, os, shutil

W = 24                                   # 24 now, 48 later
DRIVE = '/content/drive/MyDrive/fyp_solar'
LOCAL = '/content/data'
os.makedirs(LOCAL, exist_ok=True)

def get(name):
    hits = glob.glob(f'{DRIVE}/{name}*')
    assert hits, f"not found: {name}"
    dst = f'{LOCAL}/{os.path.basename(hits[0])}'
    if not os.path.exists(dst):
        shutil.copy(hits[0], dst)
    return dst

paths = {k: get(n) for k, n in {
    'Xtr': f'train_X_{W}h_v2.npy', 'ytr': f'train_y_{W}h_v2.npy',
    'Xva': f'val_X_{W}h_v2.npy',   'yva': f'val_y_{W}h_v2.npy',
    'Xte': f'test_X_{W}h_v2.npy',  'yte': f'test_y_{W}h_v2.npy',
    'mte': f'test_meta_{W}h',      'stats': 'solar_scaler_stats'}.items()}
print("copied")

Mounted at /content/drive
copied


In [2]:
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
keras.utils.set_random_seed(42)
print(tf.config.list_physical_devices('GPU'))     # should show a GPU

Xtr, ytr = np.load(paths['Xtr'], mmap_mode='r'), np.load(paths['ytr'])
Xva, yva = np.load(paths['Xva'], mmap_mode='r'), np.load(paths['yva'])
Xte, yte = np.load(paths['Xte'], mmap_mode='r'), np.load(paths['yte'])
print(Xtr.shape, Xva.shape, Xte.shape)

class Windows(keras.utils.Sequence):
    def __init__(self, X, y, bs=256, shuffle=False):
        super().__init__()
        self.X, self.y, self.bs, self.shuffle = X, y, bs, shuffle
        self.idx = np.arange(len(X))
        self.on_epoch_end()
    def __len__(self):
        return int(np.ceil(len(self.X) / self.bs))
    def __getitem__(self, i):
        b = np.sort(self.idx[i * self.bs:(i + 1) * self.bs])
        return np.asarray(self.X[b]), np.asarray(self.y[b])
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.idx)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
(2770824, 24, 7) (741034, 24, 7) (735451, 24, 7)


In [3]:
model = keras.Sequential([
    keras.layers.Input(shape=(W, 7)),
    keras.layers.LSTM(64),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')

early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(Windows(Xtr, ytr, 256, shuffle=True), validation_data=Windows(Xva, yva, 1024),
          epochs=20, callbacks=[early], verbose=2)
model.save(f'{DRIVE}/lstm_{W}h_baseline.keras')

Epoch 1/20
10824/10824 - 79s - 7ms/step - loss: 0.0227 - val_loss: 0.0136
Epoch 2/20
10824/10824 - 64s - 6ms/step - loss: 0.0184 - val_loss: 0.0140
Epoch 3/20
10824/10824 - 62s - 6ms/step - loss: 0.0180 - val_loss: 0.0127
Epoch 4/20
10824/10824 - 63s - 6ms/step - loss: 0.0178 - val_loss: 0.0128
Epoch 5/20
10824/10824 - 64s - 6ms/step - loss: 0.0176 - val_loss: 0.0127
Epoch 6/20
10824/10824 - 64s - 6ms/step - loss: 0.0175 - val_loss: 0.0124
Epoch 7/20
10824/10824 - 63s - 6ms/step - loss: 0.0174 - val_loss: 0.0129
Epoch 8/20
10824/10824 - 64s - 6ms/step - loss: 0.0173 - val_loss: 0.0127
Epoch 9/20
10824/10824 - 64s - 6ms/step - loss: 0.0173 - val_loss: 0.0124
Epoch 10/20
10824/10824 - 64s - 6ms/step - loss: 0.0172 - val_loss: 0.0123
Epoch 11/20
10824/10824 - 64s - 6ms/step - loss: 0.0172 - val_loss: 0.0122
Epoch 12/20
10824/10824 - 85s - 8ms/step - loss: 0.0172 - val_loss: 0.0122
Epoch 13/20
10824/10824 - 64s - 6ms/step - loss: 0.0171 - val_loss: 0.0122
Epoch 14/20
10824/10824 - 65s - 6m

In [4]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pred = model.predict(Windows(Xte, yte, 4096), verbose=0).ravel()
df = pd.read_csv(paths['mte'])
df['pred_s'], df['true_s'] = pred, yte
df = df.merge(pd.read_csv(paths['stats']), on='place', how='left')
df['pred'] = df['pred_s'] * df['solar_std'] + df['solar_mean']
df['true'] = df['true_s'] * df['solar_std'] + df['solar_mean']

rmse = np.sqrt(mean_squared_error(df['true'], df['pred']))
mae = mean_absolute_error(df['true'], df['pred'])
day = df['true'] > 10
mape = (np.abs(df.loc[day, 'true'] - df.loc[day, 'pred']) / df.loc[day, 'true']).mean() * 100
print(f"LSTM {W}h | RMSE {rmse:.2f} | MAE {mae:.2f} | R2 {r2_score(df['true'], df['pred']):.4f} "
      f"| nRMSE {rmse / df['true'].mean() * 100:.2f}% | MAPE {mape:.2f}%")

df[['place', 'datetime', 'pred', 'true']].to_csv(f'{DRIVE}/lstm_{W}h_test_predictions.csv', index=False)

LSTM 24h | RMSE 30.43 | MAE 15.10 | R2 0.9880 | nRMSE 14.74% | MAPE 10.75%


In [5]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd
DRIVE = '/content/drive/MyDrive/fyp_solar'
WINDOWS = [24, 48]

dfs = []
for W_ in WINDOWS:
    d = pd.read_csv(f'{DRIVE}/lstm_{W_}h_test_predictions.csv', usecols=['place','datetime','pred','true'])
    dfs.append(d.rename(columns={'pred': f'pred_{W_}', 'true': f'true_{W_}'}))

c = dfs[0].merge(dfs[1], on=['place','datetime'])
print('rows per run:', [len(d) for d in dfs], '| common rows:', len(c))
print('max |true diff| (should be ~0):', (c['true_24'] - c['true_48']).abs().max())

def station_avg_metrics(place, p, t, day_thr=50):
    df = pd.DataFrame({'place': place, 'p': np.clip(p, 0, None), 't': t})
    def f(g):
        e = g.p - g.t; rmse = np.sqrt((e**2).mean()); day = g.t > day_thr
        return pd.Series({'RMSE': rmse, 'MAE': e.abs().mean(),
                          'R2': 1 - (e**2).sum() / ((g.t - g.t.mean())**2).sum(),
                          'nRMSE': rmse / g.t.mean(),
                          'MAPE_day%': (e[day].abs() / g.t[day]).mean() * 100})
    return df.groupby('place')[['p','t']].apply(f).mean()

out = pd.DataFrame({f'{w}h': station_avg_metrics(c['place'], c[f'pred_{w}'], c['true_24'])
                    for w in WINDOWS}).round(3)
print(out)
out.to_csv(f'{DRIVE}/window_comparison_lstm.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rows per run: [735451, 715219] | common rows: 715219
max |true diff| (should be ~0): 0.0
              24h     48h
RMSE       29.770  29.423
MAE        14.893  14.636
R2          0.988   0.988
nRMSE       0.146   0.144
MAPE_day%   8.555   8.632


In [7]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil, numpy as np, pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

DRIVE = '/content/drive/MyDrive/fyp_solar'
LOCAL = '/content/data'
os.makedirs(LOCAL, exist_ok=True)

def load_base(split):
    X = np.load(f'{DRIVE}/base_{split}_X.npy')
    key = pd.read_csv(f'{DRIVE}/base_{split}_key.csv.gz', parse_dates=['datetime'])
    return X, key

def valid_starts(key, W):
    """Window start rows with W+1 consecutive hourly rows (W input hours + 1 target hour) at one station."""
    pc = pd.factorize(key['place'])[0]
    t = key['datetime'].values.astype('datetime64[s]').astype('int64')
    brk = np.r_[True, (pc[1:] != pc[:-1]) | (np.diff(t) != 3600)]
    seg_id = np.cumsum(brk) - 1
    seg_start = np.flatnonzero(brk)
    seg_len = np.bincount(seg_id)
    pos = np.arange(len(key)) - seg_start[seg_id]
    after = seg_len[seg_id] - pos - 1
    return np.flatnonzero(after >= W)

# ---------- self-check: rebuild the 24h TEST windows ----------
Xb, key = load_base('test')
s = valid_starts(key, 24)
new_keys = key.iloc[s + 24][['place', 'datetime']]
old = pd.read_csv(glob.glob(f'{DRIVE}/test_meta_24h*')[0], parse_dates=['datetime'])
y_old = np.load(f'{DRIVE}/test_y_24h_v2.npy')
y_new = Xb[s + 24, 6]
print('windows rebuilt:', len(s), '| existing:', len(old))
print('same station+time set:', set(zip(new_keys.place, new_keys.datetime)) == set(zip(old.place, old.datetime)))
print('same y values:', len(y_new) == len(y_old) and np.allclose(np.sort(y_new), np.sort(y_old), atol=1e-5))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
windows rebuilt: 735451 | existing: 735451
same station+time set: True
same y values: True


In [8]:
def build(split, W, chunk=50_000):
    Xb, key = load_base(split)
    s = valid_starts(key, W)
    n = len(s)
    out = np.lib.format.open_memmap(f'{LOCAL}/{split}_X_{W}h_v2.npy', mode='w+',
                                    dtype='float32', shape=(n, W, 7))
    sw = sliding_window_view(Xb, W, axis=0)               # view, no copy: (rows-W+1, 7, W)
    for a in range(0, n, chunk):
        out[a:a + chunk] = sw[s[a:a + chunk]].transpose(0, 2, 1)
    out.flush(); del out
    np.save(f'{LOCAL}/{split}_y_{W}h_v2.npy', Xb[s + W, 6].astype('float32'))
    if split == 'test':
        key.iloc[s + W][['place', 'datetime']].to_csv(f'{LOCAL}/test_meta_{W}h.csv', index=False)
    print(split, f'{W}h:', n, 'windows')

for sp in ['train', 'val', 'test']:
    build(sp, 72)
shutil.copy(glob.glob(f'{DRIVE}/solar_scaler_stats*')[0], f'{LOCAL}/solar_scaler_stats.xls')

train 72h: 2148803 windows
val 72h: 703331 windows
test 72h: 697537 windows


'/content/data/solar_scaler_stats.xls'

In [9]:
W = 72
LOCAL = '/content/data'
DRIVE = '/content/drive/MyDrive/fyp_solar'
paths = {
    'Xtr': f'{LOCAL}/train_X_{W}h_v2.npy', 'ytr': f'{LOCAL}/train_y_{W}h_v2.npy',
    'Xva': f'{LOCAL}/val_X_{W}h_v2.npy',   'yva': f'{LOCAL}/val_y_{W}h_v2.npy',
    'Xte': f'{LOCAL}/test_X_{W}h_v2.npy',  'yte': f'{LOCAL}/test_y_{W}h_v2.npy',
    'mte': f'{LOCAL}/test_meta_{W}h.csv',  'stats': f'{LOCAL}/solar_scaler_stats.xls'}
print({k: os.path.exists(v) for k, v in paths.items()})

{'Xtr': True, 'ytr': True, 'Xva': True, 'yva': True, 'Xte': True, 'yte': True, 'mte': True, 'stats': True}


In [10]:
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
keras.utils.set_random_seed(42)
print(tf.config.list_physical_devices('GPU'))     # should show a GPU

Xtr, ytr = np.load(paths['Xtr'], mmap_mode='r'), np.load(paths['ytr'])
Xva, yva = np.load(paths['Xva'], mmap_mode='r'), np.load(paths['yva'])
Xte, yte = np.load(paths['Xte'], mmap_mode='r'), np.load(paths['yte'])
print(Xtr.shape, Xva.shape, Xte.shape)

class Windows(keras.utils.Sequence):
    def __init__(self, X, y, bs=256, shuffle=False):
        super().__init__()
        self.X, self.y, self.bs, self.shuffle = X, y, bs, shuffle
        self.idx = np.arange(len(X))
        self.on_epoch_end()
    def __len__(self):
        return int(np.ceil(len(self.X) / self.bs))
    def __getitem__(self, i):
        b = np.sort(self.idx[i * self.bs:(i + 1) * self.bs])
        return np.asarray(self.X[b]), np.asarray(self.y[b])
    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.idx)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
(2148803, 72, 7) (703331, 72, 7) (697537, 72, 7)


In [11]:
model = keras.Sequential([
    keras.layers.Input(shape=(W, 7)),
    keras.layers.LSTM(64),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')

early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(Windows(Xtr, ytr, 256, shuffle=True), validation_data=Windows(Xva, yva, 1024),
          epochs=20, callbacks=[early], verbose=2)
model.save(f'{DRIVE}/lstm_{W}h_baseline.keras')

Epoch 1/20
8394/8394 - 97s - 12ms/step - loss: 0.0240 - val_loss: 0.0133
Epoch 2/20
8394/8394 - 140s - 17ms/step - loss: 0.0186 - val_loss: 0.0129
Epoch 3/20
8394/8394 - 68s - 8ms/step - loss: 0.0181 - val_loss: 0.0130
Epoch 4/20
8394/8394 - 68s - 8ms/step - loss: 0.0178 - val_loss: 0.0126
Epoch 5/20
8394/8394 - 65s - 8ms/step - loss: 0.0176 - val_loss: 0.0127
Epoch 6/20
8394/8394 - 64s - 8ms/step - loss: 0.0174 - val_loss: 0.0127
Epoch 7/20
8394/8394 - 70s - 8ms/step - loss: 0.0173 - val_loss: 0.0124
Epoch 8/20
8394/8394 - 72s - 9ms/step - loss: 0.0173 - val_loss: 0.0123
Epoch 9/20
8394/8394 - 65s - 8ms/step - loss: 0.0172 - val_loss: 0.0131
Epoch 10/20
8394/8394 - 64s - 8ms/step - loss: 0.0171 - val_loss: 0.0123
Epoch 11/20
8394/8394 - 66s - 8ms/step - loss: 0.0172 - val_loss: 0.0126
Epoch 12/20
8394/8394 - 67s - 8ms/step - loss: 0.0171 - val_loss: 0.0124
Epoch 13/20
8394/8394 - 64s - 8ms/step - loss: 0.0170 - val_loss: 0.0123
Epoch 14/20
8394/8394 - 65s - 8ms/step - loss: 0.0170 - v

In [12]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pred = model.predict(Windows(Xte, yte, 4096), verbose=0).ravel()
df = pd.read_csv(paths['mte'])
df['pred_s'], df['true_s'] = pred, yte
df = df.merge(pd.read_csv(paths['stats']), on='place', how='left')
df['pred'] = df['pred_s'] * df['solar_std'] + df['solar_mean']
df['true'] = df['true_s'] * df['solar_std'] + df['solar_mean']

rmse = np.sqrt(mean_squared_error(df['true'], df['pred']))
mae = mean_absolute_error(df['true'], df['pred'])
day = df['true'] > 10
mape = (np.abs(df.loc[day, 'true'] - df.loc[day, 'pred']) / df.loc[day, 'true']).mean() * 100
print(f"LSTM {W}h | RMSE {rmse:.2f} | MAE {mae:.2f} | R2 {r2_score(df['true'], df['pred']):.4f} "
      f"| nRMSE {rmse / df['true'].mean() * 100:.2f}% | MAPE {mape:.2f}%")

df[['place', 'datetime', 'pred', 'true']].to_csv(f'{DRIVE}/lstm_{W}h_test_predictions.csv', index=False)

LSTM 72h | RMSE 30.15 | MAE 14.98 | R2 0.9883 | nRMSE 14.60% | MAPE 10.77%


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd
DRIVE = '/content/drive/MyDrive/fyp_solar'
WINDOWS = [24, 48, 72]

dfs = []
for W_ in WINDOWS:
    d = pd.read_csv(f'{DRIVE}/lstm_{W_}h_test_predictions.csv', usecols=['place','datetime','pred','true'])
    d['datetime'] = pd.to_datetime(d['datetime'])
    dfs.append(d.rename(columns={'pred': f'pred_{W_}', 'true': f'true_{W_}'}))

c = dfs[0]
for d in dfs[1:]:
    c = c.merge(d, on=['place','datetime'])
print('rows per run:', [len(d) for d in dfs], '| common rows:', len(c))
print('max |true diff| (should be ~0):',
      max((c['true_24'] - c[f'true_{w}']).abs().max() for w in WINDOWS[1:]))

def station_avg_metrics(place, p, t, day_thr=50):
    df = pd.DataFrame({'place': place, 'p': np.clip(p, 0, None), 't': t})
    def f(g):
        e = g.p - g.t; rmse = np.sqrt((e**2).mean()); day = g.t > day_thr
        return pd.Series({'RMSE': rmse, 'MAE': e.abs().mean(),
                          'R2': 1 - (e**2).sum() / ((g.t - g.t.mean())**2).sum(),
                          'nRMSE': rmse / g.t.mean(),
                          'MAPE_day%': (e[day].abs() / g.t[day]).mean() * 100})
    return df.groupby('place')[['p','t']].apply(f).mean()

out = pd.DataFrame({f'{w}h': station_avg_metrics(c['place'], c[f'pred_{w}'], c['true_24'])
                    for w in WINDOWS}).round(3)
print(out)
out.to_csv(f'{DRIVE}/window_comparison_lstm_3windows.csv')

Mounted at /content/drive
rows per run: [735451, 715219, 697537] | common rows: 697537
max |true diff| (should be ~0): 0.0
              24h     48h     72h
RMSE       29.767  29.373  29.236
MAE        14.939  14.663  14.538
R2          0.988   0.988   0.988
nRMSE       0.147   0.145   0.145
MAPE_day%   8.745   8.796   8.430
